# Zerobus Ingest (HTTP REST, sync)

- clone to your workspace
- click Run All
  
## Data path — no customer-managed queue

```
Producer                        Databricks                   Consumer    
+----------+          +---------------------------+          +----------+
| App w/   |  HTTP    | Zerobus     Unity Catalog | full DML | AI       |
| requests |--------->| REST insert -> Delta      |<---------| BI       |
| POST     |          |                           |          | Agents   |
+----------+          +---------------------------+          +----------+
```

*Same Unity Catalog target as the gRPC notebook; this path uses the **HTTP insert API** instead of `ZerobusSdk` streams.*

## Setup 

- Auto-configure Zerobus endpoint 
- Auto-configure Service Principal (admin priv required to create SP if not present)
- Catalog, Schema, Table permissions

## Show performance characteristics
  - **~400 ms/row** for single-row inserts (serial POST; varies with client-to-endpoint distance)
  - **greater than ~250 ms+** for whole-batch insert
  - **~5 s** until all rows are visible

## Show system tables
  - system.lakeflow.zerobus_ingest (rows and bytes)
  - system.lakeflow.zerobus_stream (connections)
---

This notebook follows **[Use the Zerobus Ingest connector](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest?language=Python%20SDK)** (REST / JSON alongside the Python SDK).

### More links
- [Zerobus overview](https://docs.databricks.com/aws/en/ingestion/zerobus-overview)
- [Python SDK repository](https://github.com/databricks/zerobus-sdk-py)
- [Get workspace URL and Zerobus endpoint](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#get-your-workspace-url-and-zerobus-ingest-endpoint)


### Step 1: Prepare the environment

In [1]:
%pip install --quiet requests


Note: you may need to restart the kernel to use updated packages.


### Step 2: Configuration

### Step 2.a: Workspace and Zerobus endpoint (auto)

**Region:** `w.metastores.summary().region` — from the Unity Catalog metastore summary. Override with `DATABRICKS_REGION_OVERRIDE` if needed.



In [2]:
import os
from databricks.sdk import WorkspaceClient

DATABRICKS_REGION_OVERRIDE = ""

_CLOUD_DOMAIN = {
    "AWS": "cloud.databricks.com",
    "AZURE": "azuredatabricks.net",
    "GCP": "gcp.databricks.com",
}

# Connect sets metadata-service auth; clear so WorkspaceClient uses ~/.databrickscfg locally.
if (os.environ.get("DATABRICKS_AUTH_TYPE") or "").strip().lower() == "metadata-service":
    os.environ.pop("DATABRICKS_AUTH_TYPE", None)
    os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
    print("Cleared metadata-service auth env vars for WorkspaceClient() (Connect-only; SDK will use your Databricks CLI / profile auth).")

_w = WorkspaceClient()
DATABRICKS_WORKSPACE_URL = _w.config.host.rstrip("/")
DATABRICKS_WORKSPACE_ID = str(_w.get_workspace_id())
DATABRICKS_WORKSPACE_O_QUERY = f"?o={DATABRICKS_WORKSPACE_ID}"
DATABRICKS_WORKSPACE_SP_UI_PREFIX = (
    f"{DATABRICKS_WORKSPACE_URL}/settings/workspace/identity-and-access/service-principals"
)

_summary = _w.metastores.summary()
DATABRICKS_REGION = (DATABRICKS_REGION_OVERRIDE or "").strip() or _summary.region
_domain = _CLOUD_DOMAIN.get((_summary.cloud or "").upper())
if not DATABRICKS_REGION or not _domain:
    raise RuntimeError(f"Unexpected metastore cloud={_summary.cloud!r} or empty region. Set DATABRICKS_REGION_OVERRIDE.")

ZEROBUS_INGEST_URL = f"https://{DATABRICKS_WORKSPACE_ID}.zerobus.{DATABRICKS_REGION}.{_domain}"
SERVER_ENDPOINT = ZEROBUS_INGEST_URL
print(f"{DATABRICKS_WORKSPACE_ID=}\n{DATABRICKS_WORKSPACE_URL=}\n{DATABRICKS_REGION=}\n{ZEROBUS_INGEST_URL=}\n{SERVER_ENDPOINT=}")


Cleared metadata-service auth env vars for WorkspaceClient() (Connect-only; SDK will use your Databricks CLI / profile auth).
DATABRICKS_WORKSPACE_ID='1444828305810485'
DATABRICKS_WORKSPACE_URL='https://e2-demo-field-eng.cloud.databricks.com'
DATABRICKS_REGION='us-west-2'
ZEROBUS_INGEST_URL='https://1444828305810485.zerobus.us-west-2.cloud.databricks.com'
SERVER_ENDPOINT='https://1444828305810485.zerobus.us-west-2.cloud.databricks.com'


### Step 2.b: Service principal + secret

- Parse **`SP_NAME`**: `scope--secretKey--jsonField`, or `prefix--scope--secretKey--jsonField`, or a **short** `prefix` with optional `--` tail. Omitted tail segments default to scope `lfczerobusdemo`, secret key `lfczerobusdemo`, then oauth json field `ZEROBUS_OAUTH_SECRET`. A 3-segment name is treated as legacy `scope--key--field` when the third segment looks like an UPPER_SNAKE json key; otherwise as `prefix--scope--key` with the default field.
- **Create the secret scope** and **secret key** if missing (empty JSON `{}` on first create).
- **Create the workspace service principal** if needed (`display_name` equals **`SP_NAME`**).
- **Write initial JSON** (`ZEROBUS_SERVICE_PRINCIPAL_*`, `ZEROBUS_APP_ID`, and the OAuth field key — empty until 2.c mints).
- Does **not** read or mint the OAuth **client secret** (that is **Step 2.c**).

Reference: `notebooks/zerobus_service_principal.ipynb`.


In [3]:
# Step 2.b: Secret scope/key + service principal + initial JSON (no OAuth client secret mint here).

# Workspace-specific prefix; optional tail: --<scope>--<secretKey>--<oauthJsonField>
SP_NAME = "lfcdemo_zerobus"

_DEFAULT_SP_SECRET_SCOPE = "lfczerobusdemo"
_DEFAULT_SP_SECRET_KEY = "lfczerobusdemo"
_DEFAULT_SP_OAUTH_JSON_FIELD = "ZEROBUS_OAUTH_SECRET"

import json
import re

from databricks.sdk.errors import NotFound, ResourceAlreadyExists, ResourceDoesNotExist


def _looks_like_oauth_json_key(s: str) -> bool:
    """Legacy 3-part refs use an UPPER_SNAKE json field as the third segment."""
    return bool(re.fullmatch(r"[A-Z][A-Z0-9_]*", s))


def _parse_secret_ref(ref: str):
    """Parse scope--secretKey--jsonField or prefix--scope--secretKey--jsonField.

    With a leading workspace prefix, trailing segments may be omitted: missing scope,
    secret key, and oauth json field default to _DEFAULT_SP_SECRET_SCOPE,
    _DEFAULT_SP_SECRET_KEY, and _DEFAULT_SP_OAUTH_JSON_FIELD respectively.

    A 3-segment value is either that legacy form (third segment looks like a json field
    name) or prefix--scope--secretKey with the default oauth json field.
    """
    parts = [p for p in str(ref).strip().split("--") if p != ""]
    if not parts:
        raise ValueError("SP_NAME is empty or only contains '--' separators.")
    if len(parts) > 4:
        raise ValueError(
            "Secret locator: at most 4 '--' segments "
            "(prefix--scope--secretKey--jsonField, or scope--secretKey--jsonField); "
            f"got {ref!r}"
        )
    if len(parts) == 1:
        return (
            _DEFAULT_SP_SECRET_SCOPE,
            _DEFAULT_SP_SECRET_KEY,
            _DEFAULT_SP_OAUTH_JSON_FIELD,
        )
    if len(parts) == 2:
        return parts[1], _DEFAULT_SP_SECRET_KEY, _DEFAULT_SP_OAUTH_JSON_FIELD
    if len(parts) == 3:
        if _looks_like_oauth_json_key(parts[2]):
            return parts[0], parts[1], parts[2]
        return parts[1], parts[2], _DEFAULT_SP_OAUTH_JSON_FIELD
    return parts[1], parts[2], parts[3]


if not str(SP_NAME).strip():
    raise ValueError("SP_NAME is empty.")

_SECRET_SCOPE, _SECRET_KEY, _oauth_field = _parse_secret_ref(SP_NAME)
_config_keys = [
    "ZEROBUS_SERVICE_PRINCIPAL_NAME",
    "ZEROBUS_SERVICE_PRINCIPAL_ID",
    "ZEROBUS_APP_ID",
    _oauth_field,
]


def _put_secret_json(blob: dict) -> None:
    """Write the full JSON value for this scope/key (shared with Step 2.c)."""
    _w.secrets.put_secret(
        scope=_SECRET_SCOPE,
        key=_SECRET_KEY,
        string_value=json.dumps(blob, indent=2),
    )


def _ensure_secret_scope_and_key() -> None:
    try:
        _w.secrets.create_scope(_SECRET_SCOPE)
        print(f"Created secret scope {_SECRET_SCOPE!r}")
    except ResourceAlreadyExists:
        pass
    try:
        _w.secrets.get_secret(_SECRET_SCOPE, _SECRET_KEY)
    except ResourceDoesNotExist:
        _w.secrets.put_secret(scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value="{}")
        print(f"Created empty secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r}")


def _load_saved() -> dict:
    try:
        raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY)
        print(f"Loaded config from secret scope={_SECRET_SCOPE!r} key={_SECRET_KEY!r}")
        return json.loads(raw)
    except Exception:
        pass
    _ensure_secret_scope_and_key()
    return {}


def _save_config(updates: dict) -> None:
    try:
        current = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
    except ResourceDoesNotExist:
        current = {}
    current.update(updates)
    _put_secret_json(current)
    print(f"Saved {list(updates.keys())} to secret {_SECRET_SCOPE}/{_SECRET_KEY}")


_saved = _load_saved()
_config = {k: _saved.get(k, "") for k in _config_keys}
_config_original = {k: _saved.get(k, "") for k in _config_keys}

_sp_state = {"replaced_stale_sp": False}


def _ensure_sp() -> None:
    sp_id = _config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")

    def _create_sp():
        _name = str(SP_NAME).strip()
        _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] = _name
        _sp = _w.service_principals.create(display_name=_name)
        _config["ZEROBUS_SERVICE_PRINCIPAL_ID"] = str(_sp.id)
        _config["ZEROBUS_APP_ID"] = str(_sp.application_id)
        print(f"Created SP {_name!r} id={_sp.id} APP_ID={_config['ZEROBUS_APP_ID']}")

    if not str(sp_id).strip():
        print("ZEROBUS_SERVICE_PRINCIPAL_ID not set — creating service principal")
        _create_sp()
        return

    try:
        _sp = _w.service_principals.get(sp_id)
        print(f"SP exists: {_sp.display_name!r} sp_id={sp_id}")
        api_app = str(_sp.application_id)
        stored_app = str(_config.get("ZEROBUS_APP_ID", "")).strip()
        if stored_app and stored_app != api_app:
            raise RuntimeError(
                f"Secret ZEROBUS_APP_ID={stored_app!r} does not match workspace SP application_id={api_app!r} "
                f"for sp_id={sp_id!r}. Fix the secret JSON or the service principal."
            )
        if not stored_app:
            _config["ZEROBUS_APP_ID"] = api_app
            print(f"Backfilled ZEROBUS_APP_ID={_config['ZEROBUS_APP_ID']}")
    except NotFound:
        print(f"SP id={sp_id!r} not found — creating a replacement")
        _sp_state["replaced_stale_sp"] = True
        _create_sp()


_ensure_sp()

_o_sp = str(_config_original.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
_o_app = str(_config_original.get("ZEROBUS_APP_ID", "")).strip()
_n_sp = str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
_n_app = str(_config.get("ZEROBUS_APP_ID", "")).strip()

# Once both ids were stored, they must not drift unless we replaced a deleted SP (stale id in the secret).
if _o_sp and _o_app and not _sp_state["replaced_stale_sp"]:
    if _n_sp != _o_sp or _n_app != _o_app:
        raise RuntimeError(
            "Refusing to persist: ZEROBUS_SERVICE_PRINCIPAL_ID or ZEROBUS_APP_ID would change. "
            f"stored sp_id={_o_sp!r} app_id={_o_app!r}; after Step 2.b got sp_id={_n_sp!r} app_id={_n_app!r}. "
            "If you rotated the service principal, update or clear the secret JSON manually."
        )

# Persist when we first record SP metadata, backfill a missing APP_ID, replace a deleted SP, or align name/oauth keys.
# Steady re-runs with the same JSON produce no updates (not an error—just no write).
_updates = {k: _config[k] for k in _config_keys if _config.get(k) != _config_original.get(k)}
if _updates:
    _save_config(_updates)

if not str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip():
    raise RuntimeError(
        "Step 2.b: ZEROBUS_SERVICE_PRINCIPAL_ID is still empty after bootstrap; cannot continue."
    )
if not str(_config.get("ZEROBUS_APP_ID", "")).strip():
    raise RuntimeError("Step 2.b: ZEROBUS_APP_ID is empty after bootstrap; cannot continue.")


Loaded config from secret scope='lfczerobusdemo' key='lfczerobusdemo'
SP exists: 'lfcdemo_zerobus_sp' sp_id=75332893425169


In [4]:
# After Step 2.b: admin UI for this service principal (DATABRICKS_WORKSPACE_* from Step 2.a).
_sp = str(_config["ZEROBUS_SERVICE_PRINCIPAL_ID"]).strip()
print(f"{DATABRICKS_WORKSPACE_SP_UI_PREFIX}/{_sp}{DATABRICKS_WORKSPACE_O_QUERY}")


https://e2-demo-field-eng.cloud.databricks.com/settings/workspace/identity-and-access/service-principals/75332893425169?o=1444828305810485


### Step 2.c: OAuth client secret

- **Read** the JSON from the same secret **Step 2.b** configured (`_SECRET_SCOPE` / `_SECRET_KEY` / `_oauth_field`).
- **Validate** the stored client secret with the workspace **OIDC** token endpoint.
- If **missing or invalid**, **mint** a new OAuth client secret, **write** the updated JSON back to the secret, then set **`CLIENT_ID`** / **`CLIENT_SECRET`** for later cells.


In [5]:
# Step 2.c: Read client secret from the Step 2.b secret; OIDC-validate; mint + persist if invalid.
# Requires Step 2.b (defines _SECRET_SCOPE, _SECRET_KEY, _oauth_field, _put_secret_json) and Step 2.a (_w).

import json
import urllib.error
import urllib.request
from urllib.parse import urlencode


def _credentials_valid(client_id: str, client_secret: str, workspace_url: str) -> bool:
    """True if client_id + client_secret work against the workspace OIDC token endpoint."""
    if not client_id or not client_secret:
        return False
    try:
        body = urlencode(
            {
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "all-apis",
            }
        ).encode()
        req = urllib.request.Request(
            f"{workspace_url.rstrip('/')}/oidc/v1/token",
            data=body,
            method="POST",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        with urllib.request.urlopen(req, timeout=15) as resp:
            return 200 <= getattr(resp, "status", 200) < 300
    except urllib.error.HTTPError as ex:
        try:
            msg = ex.read().decode(errors="replace")
        except Exception:
            msg = str(ex)
        print(f"OIDC check: HTTP {ex.code} {msg[:500]}")
        return False
    except Exception as ex:
        print(f"OIDC check: {ex}")
        return False


for _need in ("_SECRET_SCOPE", "_SECRET_KEY", "_oauth_field", "_put_secret_json"):
    if _need not in globals():
        raise RuntimeError(
            f"Step 2.c requires Step 2.b first (missing {_need}). Re-run Step 2.b after restarting the kernel if needed."
        )

_blob = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
if "ZEROBUS_APP_ID" not in _blob:
    raise KeyError(
        "Secret JSON must include ZEROBUS_APP_ID (OAuth application id). Run Step 2.b to create the service principal."
    )
if _oauth_field not in _blob:
    _blob[_oauth_field] = ""

client_id = str(_blob["ZEROBUS_APP_ID"]).strip()
client_secret = str(_blob.get(_oauth_field, "") or "").strip()
_ws_url = _w.config.host.rstrip("/")

if _credentials_valid(client_id, client_secret, _ws_url):
    print('CLIENT_SECRET="***" OAuth client secret is valid (OIDC client_credentials).')
else:
    print("OAuth client secret missing or invalid — minting a new client secret…")
    sp_id = str(_blob.get("ZEROBUS_SERVICE_PRINCIPAL_ID") or "").strip()
    if not sp_id:
        raise RuntimeError(
            "Cannot mint OAuth client secret: ZEROBUS_SERVICE_PRINCIPAL_ID missing in secret JSON. Run Step 2.b."
        )
    _secret_obj = None
    try:
        _secret_obj = _w.service_principal_secrets_proxy.create(service_principal_id=sp_id)
    except AttributeError:
        try:
            from databricks.sdk import ServicePrincipalSecretsAPI

            _secret_obj = ServicePrincipalSecretsAPI(_w.api_client).create(service_principal_id=sp_id)
        except Exception:
            _resp = _w.api_client.do(
                "POST",
                f"/api/2.0/accounts/servicePrincipals/{sp_id}/credentials/secrets",
            )
            _blob[_oauth_field] = _resp["secret"]

    if _secret_obj is not None:
        _blob[_oauth_field] = _secret_obj.secret

    client_secret = str(_blob.get(_oauth_field, "") or "").strip()
    if not _credentials_valid(client_id, client_secret, _ws_url):
        raise RuntimeError(
            "New client secret failed OIDC check; verify workspace URL and SP permissions."
        )
    _put_secret_json(_blob)
    print(
        f"Saved new OAuth client secret to secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r} (field {_oauth_field!r})."
    )

CLIENT_ID = client_id
CLIENT_SECRET = client_secret
if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Step 2.c: CLIENT_ID or CLIENT_SECRET is empty after load/mint.")

print(f"{CLIENT_ID=}")


CLIENT_SECRET="***" OAuth client secret is valid (OIDC client_credentials).
CLIENT_ID='e6d2f259-8c72-4ac4-826b-f773af1528fc'


In [6]:
# After Step 2.c: SP secrets (OAuth) in workspace UI.
print(
    f"{DATABRICKS_WORKSPACE_SP_UI_PREFIX}/{str(_config['ZEROBUS_SERVICE_PRINCIPAL_ID']).strip()}/secrets{DATABRICKS_WORKSPACE_O_QUERY}"
)


https://e2-demo-field-eng.cloud.databricks.com/settings/workspace/identity-and-access/service-principals/75332893425169/secrets?o=1444828305810485


### Step 2.d: Catalog, schema, and table

Set `CATALOG`, `SCHEMA`, and `TABLE` (doc-style example: `main.<schema>.airquality_http_sync`). Blank **`CATALOG`** uses the Spark session default, but if that is **`hive_metastore`** (legacy / default storage), the notebook picks another catalog from `SHOW CATALOGS` — **Zerobus only supports [Unity Catalog managed Delta tables](https://docs.databricks.com/gcp/en/ingestion/zerobus-limits)** (not default storage; error **4024** if wrong). Blank **`SCHEMA`** defaults to a UC-safe name from the current Databricks user (e.g. `robert.lee@…` → `robert_lee`). Builds `TABLE_NAME` for later steps. Uses `spark`, `_w`, and `re` from Step 2.a; **`CLIENT_ID`** / **`CLIENT_SECRET`** come from **Step 2.c**.


In [7]:
# Table (Step 2.d). Default table segment: airquality_http_sync.
# Zerobus requires a UC *managed* Delta table — not hive_metastore / default storage (gRPC 4024).
CATALOG = ""  # blank → Spark default, avoiding hive_metastore when possible; or set explicitly, e.g. "main"
SCHEMA = ""  # blank → UC schema from Databricks username (e.g. robert.lee@… → robert_lee)
TABLE = "airquality_http_sync"

_ZB_EXCLUDED_CATALOGS = frozenset({"", "hive_metastore", "spark_catalog"})


def _catalog_name_from_show_row(row) -> str:
    if hasattr(row, "catalogName"):
        return str(row.catalogName)
    d = row.asDict(recursive=True)
    for k in ("catalogName", "catalog", "namespace"):
        if k in d and d[k] is not None:
            return str(d[k])
    return str(row[0])


def _default_catalog_for_zerobus() -> str:
    cur = spark.sql("SELECT current_catalog()").collect()[0][0]
    if str(cur).strip().lower() not in _ZB_EXCLUDED_CATALOGS:
        return str(cur)
    names = []
    for row in spark.sql("SHOW CATALOGS").collect():
        n = _catalog_name_from_show_row(row).strip()
        if n.lower() not in _ZB_EXCLUDED_CATALOGS:
            names.append(n)
    if not names:
        raise RuntimeError(
            "Zerobus Ingest needs a Unity Catalog catalog (managed storage). current_catalog() is "
            f"{cur!r} and no other catalogs are listed. Set CATALOG explicitly in this cell."
        )
    _main = next((x for x in names if x.lower() == "main"), None)
    if _main is not None:
        print(f"current_catalog was {cur!r}; using {_main!r} for UC-managed storage (Zerobus).")
        return _main
    names.sort(key=str.lower)
    print(f"current_catalog was {cur!r}; using {names[0]!r} for UC-managed storage (Zerobus).")
    return names[0]


if not str(CATALOG).strip():
    CATALOG = _default_catalog_for_zerobus()
    print(f"CATALOG default: {CATALOG!r}")

if not str(SCHEMA).strip():
    SCHEMA = re.sub(r"[^a-z0-9]", "_", _w.current_user.me().user_name.split("@")[0].lower())
    print(f"SCHEMA default (from current user): {SCHEMA!r}")

TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"


CATALOG default: 'main'
SCHEMA default (from current user): 'robert_lee'


### Step 3: Create your target table

Schema matches the [docs JSON example](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest?language=Python%20SDK). The next code cell creates the schema and a **managed** Delta table (`USING DELTA`) with fully qualified names, then grants the service principal **USE CATALOG**, **USE SCHEMA**, **SELECT**, and **MODIFY** in that order ([docs](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#create-a-service-principal-and-grant-permissions)). If you still see Zerobus **Unsupported table kind / default storage (4024)**, the table may have been created earlier under `hive_metastore` — `DROP TABLE` it and re-run this step after setting **`CATALOG`** in Step 2.d to a UC catalog.


In [8]:
# DDL then SP grants (grants require schema + table to exist).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
  device_name STRING,
  temp        INT,
  humidity    BIGINT
) USING DELTA
""")
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `" + CLIENT_ID + "`;").collect()
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `" + CLIENT_ID + "`;").collect()
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {TABLE_NAME} TO `" + CLIENT_ID + "`;").collect()


[]

In [9]:
# Open this table in workspace Data Explorer (Unity Catalog).
from urllib.parse import quote

print(
    f"{DATABRICKS_WORKSPACE_URL}/explore/data/"
    f"{quote(str(CATALOG), safe='')}/{quote(str(SCHEMA), safe='')}/{quote(str(TABLE), safe='')}"
    f"{DATABRICKS_WORKSPACE_O_QUERY}"
)


https://e2-demo-field-eng.cloud.databricks.com/explore/data/main/robert_lee/airquality_http_sync?o=1444828305810485


### Step 4: HTTP REST ingest (sync)

Same **`TABLE_NAME`** and OAuth **`CLIENT_ID`** / **`CLIENT_SECRET`** as the gRPC notebook, but rows are sent with **`requests.post`** to **`{ZEROBUS_INGEST_URL}/zerobus/v1/tables/{TABLE_NAME}/insert`** ([REST / JSON pattern](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest)). OIDC token uses **`resource=api://databricks/workspaces/{workspace_id}/zerobusDirectWriteApi`** and **`authorization_details`** for UC privileges (see `notebooks/zerobus_rest.ipynb`).

Run **4 setup** (token + helpers), then **4a** (up to 10 **single-row** POSTs), **4b** (one **bulk** POST for the rest), then **4c** (lakehouse visibility + timing).


In [10]:
import json
import time

import requests
from requests.auth import HTTPBasicAuth

_n = 1000  # same as gRPC doc example; lower for a quick test

_row_before = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").collect()[0]["c"]
_bounds_before = spark.sql(
    f"SELECT MIN(device_name) AS lo, MAX(device_name) AS hi FROM {TABLE_NAME}"
).collect()[0]
print(
    f"Before ingest: count={_row_before} min(device_name)={_bounds_before['lo']!r} max(device_name)={_bounds_before['hi']!r}"
)

_authorization_details = json.dumps(
    [
        {
            "type": "unity_catalog_privileges",
            "privileges": ["USE CATALOG"],
            "object_type": "CATALOG",
            "object_full_path": CATALOG,
        },
        {
            "type": "unity_catalog_privileges",
            "privileges": ["USE SCHEMA"],
            "object_type": "SCHEMA",
            "object_full_path": f"{CATALOG}.{SCHEMA}",
        },
        {
            "type": "unity_catalog_privileges",
            "privileges": ["SELECT", "MODIFY"],
            "object_type": "TABLE",
            "object_full_path": TABLE_NAME,
        },
    ]
)

_http_session = requests.Session()

_token_resp = _http_session.post(
    f"{DATABRICKS_WORKSPACE_URL}/oidc/v1/token",
    auth=HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET),
    data={
        "grant_type": "client_credentials",
        "scope": "all-apis",
        "resource": f"api://databricks/workspaces/{DATABRICKS_WORKSPACE_ID}/zerobusDirectWriteApi",
        "authorization_details": _authorization_details,
    },
    timeout=60,
)
_token_resp.raise_for_status()
_access_token = _token_resp.json()["access_token"]
print("Fetched OAuth access token for Zerobus REST (zerobusDirectWriteApi).")

_rest_insert_url = f"{ZEROBUS_INGEST_URL}/zerobus/v1/tables/{TABLE_NAME}/insert"


def _json_payload_bytes(obj: dict) -> int:
    return len(json.dumps(obj, separators=(",", ":")).encode("utf-8"))


def _rest_insert_rows(rows: list) -> tuple[requests.Response, float]:
    """POST JSON array of row objects via the shared session (keep-alive); returns (response, wall_seconds)."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {_access_token}",
    }
    t0 = time.perf_counter()
    resp = _http_session.post(
        _rest_insert_url,
        headers=headers,
        data=json.dumps(rows),
        timeout=120,
    )
    return resp, time.perf_counter() - t0


_singles = min(10, _n)
_batch_runs = 10  # consecutive batch sends for 4b latency distribution


Before ingest: count=6000 min(device_name)='sensor-0' max(device_name)='sensor-999'
Fetched OAuth access token for Zerobus REST (zerobusDirectWriteApi).


#### 4a. Single-row HTTP inserts (first up to 10 rows)

Each row is its own **`POST`** with a one-element JSON array (mirrors single-record gRPC sends). Check **`response.raise_for_status()`** on failure.


In [11]:
_row_http_seconds = []
_singles_wall_s = 0.0
_bytes_4a = 0

t_4a0 = time.perf_counter()
for i in range(_singles):
    record_dict = {
        "device_name": f"sensor-{i}",
        "temp": 20 + i % 15,
        "humidity": 50 + i % 40,
    }
    _bytes_4a += _json_payload_bytes(record_dict)
    t_row = time.perf_counter()
    resp, _ = _rest_insert_rows([record_dict])
    resp.raise_for_status()
    _row_http_seconds.append(time.perf_counter() - t_row)
_singles_wall_s = time.perf_counter() - t_4a0

# HTTP: full round-trip is both send and ack; no separate wait
_row_send_seconds = list(_row_http_seconds)
_row_wait_seconds = [0.0] * _singles   # no separate ack in HTTP
_row_ack_seconds = list(_row_http_seconds)


#### 4b. Bulk HTTP insert (remaining rows)

If **`_n > 10`**, `_batch_runs` consecutive **`POST`** requests each with a JSON array of all remaining rows, to capture latency distribution (min/mean/median/max). No gRPC stream to close.


In [12]:
_batch_lo = _singles
_batch_run_seconds = []    # wall time per batch run (full POST = send+ack combined)
_batch_send_seconds = []   # same as _batch_run_seconds (HTTP: no separate ack)
_batch_wait_seconds = []   # always 0.0 per run (no separate ack in HTTP)
_bytes_4b = 0
if _batch_lo < _n:
    batch_records = [
        {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40,
        }
        for i in range(_batch_lo, _n)
    ]
    _bytes_4b = sum(_json_payload_bytes(r) for r in batch_records)
    for _ in range(_batch_runs):
        t_4b0 = time.perf_counter()
        resp, _ = _rest_insert_rows(batch_records)
        resp.raise_for_status()
        _batch_run_s = time.perf_counter() - t_4b0
        _batch_run_seconds.append(_batch_run_s)
        _batch_send_seconds.append(_batch_run_s)   # HTTP: full POST is send+ack combined
        _batch_wait_seconds.append(0.0)            # no separate ack

_http_session.close()
_t_after_close = time.perf_counter()  # end of all inserts (no stream.close for HTTP)
_stream_open_s = None   # no gRPC stream
_stream_close_s = None

_batch_n = max(0, _n - _singles)
_total_rows_inserted = _singles + _batch_n * len(_batch_run_seconds)
_ingest_4a4b_s = _singles_wall_s + sum(_batch_run_seconds)
_bytes_payload_total = _bytes_4a + _bytes_4b


#### 4c. Lakehouse visibility

Poll `COUNT(*)` until ingested rows are visible in UC.


In [13]:
# Lakehouse visibility: table can lag a few seconds after SDK acks complete

_target_count = _row_before + _total_rows_inserted
_poll_deadline = _t_after_close + 120.0
_row_visible = _row_before
while time.perf_counter() < _poll_deadline:
    _row_visible = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").collect()[0]["c"]
    if _row_visible >= _target_count:
        break
    time.sleep(0.15)

_t_visible = time.perf_counter()
_visibility_s = _t_visible - _t_after_close        # end of ingest → COUNT(*) target
_visibility_from_first_send_s = _t_visible - t_4a0  # first row send → COUNT(*) target


#### 4d. Key metrics

In [14]:
from statistics import mean, median

print(
    f"Before ingest: count={_row_before} min(device_name)={_bounds_before['lo']!r} max(device_name)={_bounds_before['hi']!r}"
)
print(f"Ingested {_n} rows into {TABLE_NAME}")
print(f"After ingest:  expected_count={_target_count} visible_count={_row_visible}")
print(
    f"  visibility: {_visibility_from_first_send_s * 1000:.1f} ms  (from first row send → COUNT(*) reached target)"
)
print(
    f"  visibility: {_visibility_s * 1000:.1f} ms  (from end of ingest → COUNT(*) reached target)"
)
if _row_visible < _target_count:
    print("  Warning: COUNT still short after poll window; re-run or raise poll budget.")

_bounds_after = spark.sql(
    f"SELECT MIN(device_name) AS lo, MAX(device_name) AS hi FROM {TABLE_NAME}"
).collect()[0]
print(
    f"After visibility: min(device_name)={_bounds_after['lo']!r} max(device_name)={_bounds_after['hi']!r}"
)

if _stream_open_s is not None:
    print(f"stream open:  {_stream_open_s * 1000:.1f} ms")
if _stream_close_s is not None:
    print(f"stream close: {_stream_close_s * 1000:.1f} ms")
print(f"  Ingest+wait wall (4a+4b only; excludes stream.close): {_ingest_4a4b_s * 1000:.1f} ms")
if _singles:
    _ms_row_4a = (_singles_wall_s / _singles) * 1000
    print(
        f"  4a ({_singles} singles): wall {_singles_wall_s * 1000:.1f} ms  (~{_ms_row_4a:.1f} ms/row amortized)"
    )
    print(
        f"      per-row send:   "
        f"min={min(_row_send_seconds)*1000:.1f} ms "
        f"mean={mean(_row_send_seconds)*1000:.1f} ms median={median(_row_send_seconds)*1000:.1f} ms "
        f"max={max(_row_send_seconds)*1000:.1f} ms"
    )
    print(
        f"      per-row wait (ack):   "
        f"min={min(_row_wait_seconds)*1000:.1f} ms "
        f"mean={mean(_row_wait_seconds)*1000:.1f} ms median={median(_row_wait_seconds)*1000:.1f} ms "
        f"max={max(_row_wait_seconds)*1000:.1f} ms"
    )
    print(
        f"      per-row total (send+wait):   "
        f"min={min(_row_ack_seconds)*1000:.1f} ms "
        f"mean={mean(_row_ack_seconds)*1000:.1f} ms median={median(_row_ack_seconds)*1000:.1f} ms "
        f"max={max(_row_ack_seconds)*1000:.1f} ms"
    )
if _batch_run_seconds and _batch_n:
    _ms_row_4b = (mean(_batch_run_seconds) / _batch_n) * 1000
    print(
        f"  4b ({_batch_n} batched, {len(_batch_run_seconds)} runs):"
        f"  wall  min={min(_batch_run_seconds)*1000:.1f} ms"
        f"  mean={mean(_batch_run_seconds)*1000:.1f} ms"
        f"  median={median(_batch_run_seconds)*1000:.1f} ms"
        f"  max={max(_batch_run_seconds)*1000:.1f} ms"
    )
    print(
        f"      send:       "
        f"min={min(_batch_send_seconds)*1000:.1f} ms  "
        f"mean={mean(_batch_send_seconds)*1000:.1f} ms  "
        f"median={median(_batch_send_seconds)*1000:.1f} ms  "
        f"max={max(_batch_send_seconds)*1000:.1f} ms"
    )
    print(
        f"      wait (ack): "
        f"min={min(_batch_wait_seconds)*1000:.1f} ms  "
        f"mean={mean(_batch_wait_seconds)*1000:.1f} ms  "
        f"median={median(_batch_wait_seconds)*1000:.1f} ms  "
        f"max={max(_batch_wait_seconds)*1000:.1f} ms"
    )
if _singles and _batch_run_seconds and _batch_n:
    _ms_row_4a = (_singles_wall_s / _singles) * 1000
    _ms_row_4b = (mean(_batch_run_seconds) / _batch_n) * 1000
    _ratio = _ms_row_4a / _ms_row_4b
    print("  batch vs single latency comparison")
    print(f"    4a ({_singles} singles) {_ms_row_4a:.1f} wall amortized ms/row")
    print(f"    4b (1 batch of {_batch_n} rows) {_ms_row_4b:.3f} wall amortized ms/row  (mean over {len(_batch_run_seconds)} runs)")
    print(f"    4a/4b = ~{_ratio:.1f}x lower ms/row")

Before ingest: count=6000 min(device_name)='sensor-0' max(device_name)='sensor-999'
Ingested 1000 rows into main.robert_lee.airquality_http_sync
After ingest:  expected_count=15910 visible_count=15910
  visibility: 6081.7 ms  (from first row send → COUNT(*) reached target)
  visibility: 1503.2 ms  (from end of ingest → COUNT(*) reached target)
After visibility: min(device_name)='sensor-0' max(device_name)='sensor-999'
  Ingest+wait wall (4a+4b only; excludes stream.close): 4563.7 ms
  4a (10 singles): wall 2568.4 ms  (~256.8 ms/row amortized)
      per-row send:   min=199.8 ms mean=256.8 ms median=201.4 ms max=757.8 ms
      per-row wait (ack):   min=0.0 ms mean=0.0 ms median=0.0 ms max=0.0 ms
      per-row total (send+wait):   min=199.8 ms mean=256.8 ms median=201.4 ms max=757.8 ms
  4b (990 batched, 10 runs):  wall  min=187.8 ms  mean=199.5 ms  median=200.6 ms  max=202.0 ms
      send:       min=187.8 ms  mean=199.5 ms  median=200.6 ms  max=202.0 ms
      wait (ack): min=0.0 ms  mean

### gRPC-only SDK features

**Ack callbacks** and **Protocol Buffers** (`RecordType.PROTO`) apply to the **gRPC Python SDK** (`zerobus.sdk.sync`), not this HTTP notebook. See **`notebooks/zerobus_grpc_sync.ipynb`** for those sections.


### Step 5: Zerobus system tables

Zerobus **ingest** and **stream** telemetry in `system.lakeflow` (Beta; see [Zerobus system tables](https://docs.databricks.com/aws/en/admin/system-tables/zerobus-ingest)). Rows can lag; tables must be enabled for your workspace.


#### 5.a. `zerobus_ingest`

All columns from `system.lakeflow.zerobus_ingest` for this table (`SELECT *`, filtered by `table_name` and `workspace_id`, `ORDER BY commit_time DESC`, `LIMIT 20`).


In [15]:
# Step 5a: commit batches (zerobus_ingest)
# there will be some delays

try:
    display(spark.sql(f"""
    SELECT *
    FROM system.lakeflow.zerobus_ingest
    WHERE table_name = '{TABLE_NAME}'
    AND workspace_id = '{DATABRICKS_WORKSPACE_ID}'
    ORDER BY commit_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_ingest is not available in this workspace.")
        print("Ask a workspace admin to enable the Zerobus Ingest system table.")
    else:
        raise


,commit_version,stream_id,workspace_id,account_id,table_id,table_name,commit_time,committed_bytes,committed_records,tags,errors
0,6,5f6c2a5c-3f0d-4aa4-b212-1e53fe5c5083,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,2026-04-15 22:15:07.691,55890,1000,[],[]
1,5,6f45f74f-07fd-425d-96d3-4af812bbe135,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,2026-04-15 22:09:12.669,55890,1000,[],[]
2,4,766ee7ef-b851-4fe1-a539-05413da62498,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,2026-04-15 19:28:55.860,55890,1000,[],[]
3,3,e4686d84-7ad8-46c9-877d-fb3cd1aa3bdb,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,2026-04-15 18:58:46.135,55890,1000,[],[]
4,2,b2185c04-98f3-4429-a63d-319ea3411379,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,2026-04-14 22:11:00.691,55890,1000,[],[]
5,1,057d6d65-9866-4f5b-9cad-5a2d7a92c1a8,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,2026-04-14 22:08:15.590,55890,1000,[],[]


#### 5.b. `zerobus_stream`

All columns from `system.lakeflow.zerobus_stream` for this table (same `table_name` / `workspace_id` filters, `LIMIT 20`).


In [16]:
# Step 5b: stream events (zerobus_stream)

try:
    display(spark.sql(f"""
    SELECT *
    FROM system.lakeflow.zerobus_stream
    WHERE table_name = '{TABLE_NAME}'
    AND workspace_id = '{DATABRICKS_WORKSPACE_ID}'
    ORDER BY event_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_stream is not available in this workspace.")
        print("Ask a workspace admin to enable the Zerobus stream system table.")
    else:
        raise


,stream_id,event_time,workspace_id,account_id,producer_id,opened_time,closed_time,table_id,table_name,protocol,data_format,errors
0,5f6c2a5c-3f0d-4aa4-b212-1e53fe5c5083,2026-04-15 22:15:03.847,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,NaN,2026-04-15 22:15:03.847,NaT,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,HTTP,JSON,[]
1,6f45f74f-07fd-425d-96d3-4af812bbe135,2026-04-15 22:09:07.387,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,NaN,2026-04-15 22:09:07.387,NaT,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,HTTP,JSON,[]
2,766ee7ef-b851-4fe1-a539-05413da62498,2026-04-15 19:28:50.558,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,NaN,2026-04-15 19:28:50.558,NaT,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,HTTP,JSON,[]
3,e4686d84-7ad8-46c9-877d-fb3cd1aa3bdb,2026-04-15 18:58:40.937,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,NaN,2026-04-15 18:58:40.937,NaT,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,HTTP,JSON,[]
4,b2185c04-98f3-4429-a63d-319ea3411379,2026-04-14 22:10:57.277,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,NaN,2026-04-14 22:10:57.277,NaT,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,HTTP,JSON,[]
5,057d6d65-9866-4f5b-9cad-5a2d7a92c1a8,2026-04-14 22:08:10.435,1444828305810485,e6e8162c-a42f-43a0-af86-312058795a14,NaN,2026-04-14 22:08:10.435,NaT,0b6558d9-3c8a-4665-b319-4ebc98ef4544,main.robert_lee.airquality_http_sync,HTTP,JSON,[]
